# Silver: Data Quality Validation

Validates silver layer data against configurable quality rules:

## Quality Checks

1. **Completeness**: Required fields must not be null
2. **Uniqueness**: Enterprise job IDs must be unique
3. **Referential integrity**: Valid source names
4. **Format validation**: URLs, dates, enums
5. **Business rules**: Title length, company name

## Actions

- **PASS**: Record proceeds to next stage
- **WARN**: Log warning, proceed with flag
- **FAIL**: Route to quarantine table

## Architecture

**Input**: `silver.silver_jobs_current`  
**Output**: Updated DQ flags, quarantine records  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks validated batches in `metadata.dq_validated_batches`
- Automatically processes all unvalidated batches
- Idempotent: safe to re-run

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unvalidated)")
dbutils.widgets.dropdown("validation_level", "standard", ["strict", "standard", "permissive"], "Validation Level")
dbutils.widgets.dropdown("quarantine_on_fail", "true", ["true", "false"], "Quarantine Failed Records")

batch_id = dbutils.widgets.get("batch_id").strip()
validation_level = dbutils.widgets.get("validation_level")
quarantine_on_fail = dbutils.widgets.get("quarantine_on_fail") == "true"

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.window import Window

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"
QUARANTINE_SCHEMA = f"{CATALOG}.quarantine"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

# Valid enum values
VALID_REMOTE_TYPES = ["REMOTE", "HYBRID", "ONSITE", "UNKNOWN"]
VALID_SOURCES = ["arbeitnow", "remotive"]  # Extend as sources are added

In [0]:
# Create metadata table to track DQ-validated batches
metadata_table = f"{METADATA_SCHEMA}.dq_validated_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  records_validated INT,
  pass_count INT,
  warn_count INT,
  fail_count INT,
  quarantined_count INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been DQ validated (batch_id + source_name should be unique)'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("records_validated", IntegerType(), True),
    StructField("pass_count", IntegerType(), True),
    StructField("warn_count", IntegerType(), True),
    StructField("fail_count", IntegerType(), True),
    StructField("quarantined_count", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# Identify unvalidated batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in current table
    all_batches_query = f"""
        SELECT DISTINCT current_batch_id as batch_id, source_name
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE is_active = true AND soft_delete_flag = false
        AND current_batch_id IS NOT NULL
        AND current_batch_id != ''
        AND source_name IS NOT NULL
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already validated batches
    validated_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unvalidated batches (in current but not in metadata)
    unvalidated_df = all_batches_df.join(
        validated_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unvalidated_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unvalidated batches", "total_validated": 0})
    
    print(f"Found {len(batches_to_process)} unvalidated batch(es) to process")

In [0]:
from pyspark.sql.window import Window

def check_completeness(df):
    """Check required fields are not null"""
    required_fields = [
        "enterprise_job_id", 
        "source_name", 
        "source_job_id",
        "company_name_norm",
        "title_normalized",
        "record_hash"
    ]
    
    for field in required_fields:
        df = df.withColumn(
            f"dq_{field}_null",
            F.when(F.col(field).isNull(), F.lit("FAIL")).otherwise(F.lit("PASS"))
        )
    
    return df

def check_uniqueness(df):
    """Check enterprise_job_id is unique"""
    window = Window.partitionBy("enterprise_job_id")
    df = df.withColumn(
        "dq_unique_job_id",
        F.when(F.count("*").over(window) > 1, F.lit("FAIL")).otherwise(F.lit("PASS"))
    )
    return df

def check_referential_integrity(df):
    """Check source_name is valid"""
    df = df.withColumn(
        "dq_valid_source",
        F.when(F.col("source_name").isin(VALID_SOURCES), F.lit("PASS")).otherwise(F.lit("FAIL"))
    )
    return df

def check_format_validation(df):
    """Validate URL and enum formats"""
    # URL validation (simple check)
    df = df.withColumn(
        "dq_valid_url",
        F.when(
            F.col("source_url").isNull() | F.col("source_url").rlike("^https?://"),
            F.lit("PASS")
        ).otherwise(F.lit("WARN"))
    )
    
    # Remote type enum
    df = df.withColumn(
        "dq_valid_remote_type",
        F.when(
            F.col("remote_type").isin(VALID_REMOTE_TYPES),
            F.lit("PASS")
        ).otherwise(F.lit("FAIL"))
    )
    
    return df

def check_business_rules(df):
    """Business logic validation"""
    # Title length (should be reasonable)
    df = df.withColumn(
        "dq_title_length",
        F.when(
            (F.length(F.col("title_raw")) >= 3) & (F.length(F.col("title_raw")) <= 500),
            F.lit("PASS")
        ).otherwise(F.lit("WARN"))
    )
    
    # Company name not empty
    df = df.withColumn(
        "dq_company_not_empty",
        F.when(
            F.length(F.trim(F.col("company_name_raw"))) > 0,
            F.lit("PASS")
        ).otherwise(F.lit("FAIL"))
    )
    
    return df

# Quality check functions defined

In [0]:
from delta.tables import DeltaTable

# Initialize tracking
total_validated = 0
total_pass = 0
total_warn = 0
total_fail = 0
total_quarantined = 0
processed_batch_count = 0
failed_batches = []

# Ensure DQ columns exist in current table
try:
    current_schema = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").schema
    current_columns = [field.name for field in current_schema.fields]
    
    missing_columns = []
    if "dq_overall_status" not in current_columns:
        missing_columns.append("ADD COLUMN dq_overall_status STRING")
    if "dq_validated_at" not in current_columns:
        missing_columns.append("ADD COLUMN dq_validated_at TIMESTAMP")
    if "dq_validation_level" not in current_columns:
        missing_columns.append("ADD COLUMN dq_validation_level STRING")
    
    if missing_columns:
        for col_def in missing_columns:
            spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_jobs_current {col_def}")
except Exception as e:
    print(f"Warning: Could not check/add DQ columns: {e}")

# Process each batch
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Validating batch {current_batch_id} ({current_source})...", end=" ")
        
        # Load records for this batch
        batch_query = f"""
            SELECT * FROM {SILVER_SCHEMA}.silver_jobs_current
            WHERE current_batch_id = '{current_batch_id}'
            AND is_active = true AND soft_delete_flag = false
        """
        
        if current_source:
            batch_query += f" AND source_name = '{current_source}'"
        
        jobs_df = spark.sql(batch_query)
        record_count = jobs_df.count()
        
        if record_count == 0:
            print("No records found")
            continue
        
        # Drop existing DQ metadata columns (from previous runs) to avoid type conflicts
        cols_to_drop = ["dq_overall_status", "dq_validated_at", "dq_validation_level"]
        for col in cols_to_drop:
            if col in jobs_df.columns:
                jobs_df = jobs_df.drop(col)
        
        # Apply all quality checks
        validated_df = jobs_df
        validated_df = check_completeness(validated_df)
        validated_df = check_uniqueness(validated_df)
        validated_df = check_referential_integrity(validated_df)
        validated_df = check_format_validation(validated_df)
        validated_df = check_business_rules(validated_df)
        
        # Collect all DQ check columns (exclude metadata columns)
        excluded_dq_cols = {"dq_overall_status", "dq_validated_at", "dq_validation_level"}
        dq_columns = [c for c in validated_df.columns if c.startswith("dq_") and c not in excluded_dq_cols]
        
        # Calculate overall status
        fail_condition = F.array_contains(F.array(*[F.col(c) for c in dq_columns]), "FAIL")
        warn_condition = F.array_contains(F.array(*[F.col(c) for c in dq_columns]), "WARN")
        
        validated_df = validated_df.withColumn(
            "dq_overall_status",
            F.when(fail_condition, F.lit("FAIL"))
             .when(warn_condition, F.lit("WARN"))
             .otherwise(F.lit("PASS"))
        )
        
        # Add validation metadata
        validated_df = validated_df.withColumn("dq_validated_at", run_timestamp)
        validated_df = validated_df.withColumn("dq_validation_level", F.lit(validation_level))
        
        # Calculate metrics
        pass_count = validated_df.where("dq_overall_status = 'PASS'").count()
        warn_count = validated_df.where("dq_overall_status = 'WARN'").count()
        fail_count = validated_df.where("dq_overall_status = 'FAIL'").count()
        
        # Quarantine failed records
        quarantined_count = 0
        if quarantine_on_fail and fail_count > 0:
            failed_records = validated_df.where("dq_overall_status = 'FAIL'")
            
            # Collect failed rule names
            failed_rules = F.concat_ws(", ", F.array(*[
                F.when(F.col(c) == "FAIL", F.lit(c.replace("dq_", "")))
                for c in dq_columns
            ]))
            
            quarantine_df = failed_records.select(
                F.expr("uuid()").alias("quarantine_id"),
                F.col("source_name"),
                F.col("source_job_id"),
                F.col("enterprise_job_id"),
                F.lit("SILVER_DQ").alias("failure_stage"),
                failed_rules.alias("failed_rule_name"),
                F.lit("ERROR").alias("severity"),
                F.to_json(F.struct(*[c for c in jobs_df.columns])).alias("payload_json"),
                run_timestamp.alias("quarantined_at"),
                F.col("current_batch_id").alias("batch_id"),
                F.lit("PENDING").alias("reprocess_status"),
                F.lit(None).cast("string").alias("reprocess_batch_id"),
                F.lit(None).cast("timestamp").alias("resolved_at")
            )
            
            quarantine_df.write.format("delta").mode("append").saveAsTable(f"{QUARANTINE_SCHEMA}.quarantine_jobs")
            quarantined_count = fail_count
        
        # Update DQ flags in current table
        dq_update_df = validated_df.select(
            "enterprise_job_id",
            "dq_overall_status",
            "dq_validated_at",
            "dq_validation_level"
        )
        
        delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
        
        delta_current.alias("cur").merge(
            dq_update_df.alias("dq"),
            "cur.enterprise_job_id = dq.enterprise_job_id"
        ).whenMatchedUpdate(
            set={
                "dq_overall_status": "dq.dq_overall_status",
                "dq_validated_at": "dq.dq_validated_at",
                "dq_validation_level": "dq.dq_validation_level"
            }
        ).execute()
        
        # Record batch processing in metadata
        # Defensive: ensure source_name is never None/null
        safe_source = current_source if current_source else "unknown"
        metadata_record = spark.createDataFrame([
            (current_batch_id, safe_source, int(record_count), int(pass_count), int(warn_count), int(fail_count), int(quarantined_count), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_validated += record_count
        total_pass += pass_count
        total_warn += warn_count
        total_fail += fail_count
        total_quarantined += quarantined_count
        processed_batch_count += 1
        
        print(f"✓ P:{pass_count} W:{warn_count} F:{fail_count}")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            safe_source = current_source if current_source else "unknown"
            failure_record = spark.createDataFrame([
                (current_batch_id, safe_source, 0, 0, 0, 0, 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nValidated {processed_batch_count} batches, {total_validated} total records")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

In [0]:
# Show DQ validation history
if processed_batch_count > 0:
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, records_validated, pass_count, warn_count, fail_count, quarantined_count, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show sample of validated records
    sample_df = spark.sql(f"""
        SELECT enterprise_job_id, source_name, company_name_norm, title_normalized, dq_overall_status, dq_validated_at
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE dq_validated_at IS NOT NULL
        ORDER BY dq_validated_at DESC
        LIMIT 10
    """)
    display(sample_df)

In [0]:
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "total_records": total_validated,
    "pass_count": total_pass,
    "warn_count": total_warn,
    "fail_count": total_fail,
    "quarantined_count": total_quarantined,
    "pass_rate": round(total_pass / total_validated * 100, 2) if total_validated > 0 else 0,
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))